In [ ]:
!pip install pandas numpy xgboost scikit-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV
import joblib

In [ ]:
df_meals = pd.read_csv('food_data.csv')
users = [
    {'id': 1, 'name': 'Vansh', 'likes': ['chicken', 'pasta', 'bread'], 'dislikes': ['tofu'], 'spice': 2},
    {'id': 2, 'name': 'Shashwat', 'likes': ['spicy', 'paneer', 'curry'], 'dislikes': ['beef'], 'spice': 5},
    {'id': 3, 'name': 'Rajasthani', 'likes': ['potato', 'paneer', 'rice'], 'dislikes': ['mushrooms'], 'spice': 2},
    {'id': 4, 'name': 'Atharva', 'likes': ['veg', 'healthy', 'lentils'], 'dislikes': ['fried'], 'spice': 3},
    {'id': 5, 'name': 'Anni', 'likes': ['noodles', 'asian', 'cheese'], 'dislikes': ['lamb'], 'spice': 3},
    {'id': 6, 'name': 'Prajjwal', 'likes': ['soup', 'comfort', 'rice'], 'dislikes': ['raw'], 'spice': 1}
]

rows = []
for user in users:
    for _, meal in df_meals.iterrows():
        score = 5.0
        ingredients = str(meal['main_ingredients']).lower()
        name = str(meal['name']).lower()
        
        for like in user['likes']:
            if like in ingredients or like in name: score += 1.5
        for dislike in user['dislikes']:
            if dislike in ingredients or dislike in name: score -= 3.0
            
        spice_diff = abs(meal['spice_level'] - user['spice'])        
        score -= (spice_diff * 1.2)
        
        if user['id'] in [4, 5] and meal['is_veg'] == 0: score -= 4.0
        if meal['prep_time_mins'] > 45 and user['id'] in [4, 5]: score -= 1.0
        
        rating = np.clip(score + np.random.normal(0, 0.5), 1, 10)
        rows.append([user['id'], meal['meal_id'], meal['prep_time_mins'], meal['ingredient_count'], meal['spice_level'], meal['is_veg'], meal['budget_tier'], rating])

df_train = pd.DataFrame(rows, columns=['user_id', 'meal_id', 'prep_time_mins', 'ingredient_count', 'spice_level', 'is_veg', 'budget_tier', 'rating'])

In [ ]:
X = df_train.drop('rating', axis=1)
y = df_train['rating']
cat_features = ['user_id', 'meal_id']
num_features = ['prep_time_mins', 'ingredient_count', 'spice_level', 'is_veg', 'budget_tier']
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('num', StandardScaler(), num_features)
])
pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', xgb.XGBRegressor(objective='reg:squarederror', n_estimators=200, max_depth=6, learning_rate=0.1))])
pipeline.fit(X, y)
joblib.dump(pipeline, 'meal_scoring_pipeline.joblib')

['meal_scoring_pipeline.joblib']

In [20]:
sample = X.iloc[[0]]
print(f"Prediction for {users[0]['name']} and {df_meals.iloc[0]['name']}: {pipeline.predict(sample)[0]:.2f}")

Prediction for Vansh and Aloo Jeera: 4.77
